# RepoGuard — Phase 2: RL Policy Training & Evaluation on Google Colab

> **Project:** RepoGuard (Final Year Engineering Project)  
> **Workstream:** Member 3 — RL Agent Development, Environment, Policy Training & Evaluation  
> **Goal:** Train risk-conditioned Deep Q-Network (DQN) and REINFORCE agents on Google Colab GPU with persistent checkpoints saved to Google Drive.

---

### Key Architecture Specifications:
- **Action Space:** Canonical 6 actions (`SEARCH`, `INSPECT`, `VERIFY`, `ANSWER`, `ANSWER_WITH_EVIDENCE`, `ABSTAIN`).
- **Observation Space:** 18-dimensional normalized state vector ($s \in [0, 1]^{18}$).
- **Reward Function:** $r = R_{\text{final}}(\lambda) - 0.2 \cdot \frac{T}{2000} - 0.01 \cdot n$, with dynamic risk penalty $\lambda \sim \text{LogUniform}(0.25, 5.0)$.
- **Persistence:** All checkpoints and evaluation outputs are written directly to `/content/drive/MyDrive/RepoGuard/` so no work is lost on disconnect.

## 1. Verify GPU Runtime
Make sure you are connected to a GPU instance:
- Click **Runtime** > **Change runtime type** > Select **T4 GPU** (or A100/V100 if available).

In [ ]:
# Check NVIDIA GPU availability
!nvidia-smi

## 2. Mount Google Drive for Persistent Storage
Mount Google Drive to persist all models, training metrics, and evaluation summaries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Create persistent directory structure in Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/RepoGuard')
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_REPORTS = DRIVE_ROOT / 'reports'

for d in [DRIVE_CHECKPOINTS / 'dqn', DRIVE_CHECKPOINTS / 'reinforce', DRIVE_REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Persistent Drive workspace ready at: {DRIVE_ROOT}")

## 3. Clone Repository or Set Workspace
Clone the RepoGuard repository from GitHub (or pull latest changes).

In [ ]:
%cd /content

# Replace with your GitHub repository URL if private/different
REPO_URL = "https://github.com/Shreyank86/Repo-Guard.git"

if not Path('/content/Repo-Guard').exists():
    !git clone {REPO_URL} /content/Repo-Guard
else:
    %cd /content/Repo-Guard
    !git pull origin main

%cd /content/Repo-Guard
!git status

## 4. Install Dependencies
Install project requirements and editable package.

In [ ]:
%cd /content/Repo-Guard
!pip install -q -r requirements.txt
!pip install -q bitsandbytes
!pip install -q -e .
print("Dependencies installed successfully!")

## 5. Fetch Repositories & Build Vector Indexes
Fetch the 5 pinned benchmark repositories (`schedule`, `python-dotenv`, `cachetools`, `python-tabulate`, `requests`) and build their dense semantic vector indexes in one automated step.

In [ ]:
%cd /content/Repo-Guard
# Clones pinned commits and builds dense MiniLM indexes for all 5 repos
!python scripts/fetch_repos.py --index
print("All 5 benchmark repositories cloned and indexed!")

## 6. Run Unit Test Verification
Run the unit test suite to verify the RL environment, cache, networks, and policy classes.

In [ ]:
%cd /content/Repo-Guard
!pytest tests/unit/test_environment.py tests/unit/test_dqn.py tests/unit/test_reinforce.py tests/unit/test_policy_serving.py -v

## 7. Train Deep Q-Network (DQN) Policy
Train the primary DQN agent over 2,000 episodes on CUDA GPU.
- Epsilon decays from 1.0 down to 0.05.
- Random risk factor $\lambda \sim \text{LogUniform}(0.25, 5.0)$ sampled per episode.
- Experience replay buffer: 20,000 transitions.
- Checkpoints saved automatically to Google Drive every 50 episodes.

In [ ]:
%cd /content/Repo-Guard

!python scripts/train_agent.py \
    --algo dqn \
    --episodes 2000 \
    --device cuda \
    --learning-rate 0.0005 \
    --batch-size 32 \
    --checkpoint-dir /content/drive/MyDrive/RepoGuard/checkpoints/dqn \
    --log-interval 50

## 8. Train REINFORCE Agent (Policy Gradient Baseline)
Train the REINFORCE baseline agent over 1,500 episodes for comparative ablation studies.

In [ ]:
%cd /content/Repo-Guard

!python scripts/train_agent.py \
    --algo reinforce \
    --episodes 1500 \
    --device cuda \
    --learning-rate 0.0005 \
    --checkpoint-dir /content/drive/MyDrive/RepoGuard/checkpoints/reinforce \
    --log-interval 50

## 9. Comprehensive Policy Evaluation Across Risk Levels ($\lambda$)
Evaluate both trained policies across 6 risk tolerance levels $\lambda \in \{0.25, 0.5, 1.0, 2.0, 3.0, 5.0\}$.

In [ ]:
%cd /content/Repo-Guard

# Evaluate DQN Policy
!python scripts/evaluate_rl.py \
    --policy /content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt \
    --split all \
    --output-csv /content/drive/MyDrive/RepoGuard/reports/eval_dqn.csv

# Evaluate REINFORCE Policy
!python scripts/evaluate_rl.py \
    --policy /content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt \
    --split all \
    --output-csv /content/drive/MyDrive/RepoGuard/reports/eval_reinforce.csv

print("Evaluation complete! Results saved to Google Drive reports directory.")

## 10. Visualize Results & Risk-Coverage Trade-Offs
Load evaluation CSV files and plot comparative risk curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dqn_df = pd.read_csv('/content/drive/MyDrive/RepoGuard/reports/eval_dqn.csv')
rf_df = pd.read_csv('/content/drive/MyDrive/RepoGuard/reports/eval_reinforce.csv')

print("=== DQN Evaluation Summary ===")
display(dqn_df)

print("\n=== REINFORCE Evaluation Summary ===")
display(rf_df)

# Plot Risk-Conditioned Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Coverage vs Lambda
axes[0].plot(dqn_df['lambda'], dqn_df['coverage_pct'], 'o-', label='DQN', color='royalblue')
axes[0].plot(rf_df['lambda'], rf_df['coverage_pct'], 's--', label='REINFORCE', color='darkorange')
axes[0].set_xlabel('Risk Penalty (lambda)')
axes[0].set_ylabel('Coverage (%)')
axes[0].set_title('Answer Coverage vs Risk Penalty')
axes[0].grid(True)
axes[0].legend()

# 2. Hallucination Rate vs Lambda
axes[1].plot(dqn_df['lambda'], dqn_df['hallucination_rate_pct'], 'o-', label='DQN', color='crimson')
axes[1].plot(rf_df['lambda'], rf_df['hallucination_rate_pct'], 's--', label='REINFORCE', color='purple')
axes[1].set_xlabel('Risk Penalty (lambda)')
axes[1].set_ylabel('Hallucination Rate (%)')
axes[1].set_title('Hallucination Rate vs Risk Penalty')
axes[1].grid(True)
axes[1].legend()

# 3. Correct Abstention on Traps vs Lambda
axes[2].plot(dqn_df['lambda'], dqn_df['correct_abstention_pct'], 'o-', label='DQN', color='forestgreen')
axes[2].plot(rf_df['lambda'], rf_df['correct_abstention_pct'], 's--', label='REINFORCE', color='teal')
axes[2].set_xlabel('Risk Penalty (lambda)')
axes[2].set_ylabel('Correct Abstention (%)')
axes[2].set_title('Trap Abstention vs Risk Penalty')
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
chart_path = '/content/drive/MyDrive/RepoGuard/reports/risk_curves_comparison.png'
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"High-resolution comparison chart saved to: {chart_path}")

## 11. Artifact Summary & Phase 3 Integration
Your trained policies are now stored safely in Google Drive:
- `/content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt`
- `/content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt`

To deploy in Phase 3 FastAPI serving, simply load via:
```python
from repoguard.agent import RepoGuardPolicy
policy = RepoGuardPolicy('checkpoints/dqn/dqn_policy_latest.pt')
action = policy.select_action(state_vector, valid_mask)
```